# Propagating multiple observables at once
The default method for propagation `.propagate()` does process them sequentially. 

If you need to propagate multiple observable at once, you can add `num_jobs: int` argument to the `.propagate()` method.

In [1]:
import pennylane as qml
from pprop import Propagator
from pprop.propagator.pruning import DeadQubitPruner, XYWeightPruner
import time

side = 8

/home/samonaco/Pauli-Propagator/.venv/lib/python3.12/site-packages/pennylane/operation.py:2622: PennyLaneDeprecationWarning: Observable is deprecated and will be removed in v0.43. A generic Operator class should be used instead. If defining an Operator, set the is_hermitian property to True. If checking if an Operator is Hermitian, check the is_hermitian property. 
  warnings.warn(


In [2]:
def hamiltonian(side: int):
    obs = []

    # Nearest-neighbor ZZ interactions
    for x in range(side):
        for y in range(side):
            i = x * side + y

            # Right neighbor
            if y < side - 1:
                j = x * side + (y + 1)
                obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

            # Down neighbor
            if x < side - 1:
                j = (x + 1) * side + y
                obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

    # Transverse-field X terms
    for i in range(side*side):
        obs.append(qml.PauliX(i))

    exps = [qml.expval(ob) for ob in obs]
    return exps

def circuit(params):
    index = 0

    # Initial RY and RX
    for q in range(side*side):
        qml.RY(params[index], wires=q)
        index += 1
        qml.RX(params[index], wires=q)
        index += 1

    # Horizontal entanglers
    for d in range(2):
        y_start = 0 if d % 2 == 0 else 1
        for x in range(side):
            for y in range(y_start, side - 1, 2):
                i = x * side + y
                j = x * side + (y + 1)
                qml.CNOT(wires=[i, j])

    # RX layer
    for q in range(side*side):
        qml.RX(params[index], wires=q)
        index += 1

    # Vertical entanglers
    for d in range(2):
        x_start = 0 if d % 2 == 0 else 1
        for y in range(side):
            for x in range(x_start, side - 1, 2):
                i = x * side + y
                j = (x + 1) * side + y
                qml.CNOT(wires=[i, j])

    # RX layer
    for q in range(side*side):
        qml.RX(params[index], wires=q)
        index += 1

    # Final RY
    for q in range(side*side):
        qml.RY(params[index], wires=q)
        index += 1
        
    return hamiltonian(side)

In [6]:
prop = Propagator(circuit)
print(f'{len(prop.observables)} observables need to be propagated') 

176 observables need to be propagated


In [7]:
time_seq_start = time.time()
prop.propagate(pruners=[DeadQubitPruner(), XYWeightPruner()])
time_seq_end = time.time()
print(f'Default: {time_seq_end - time_seq_start:.3f} seconds')

Default: 13.882 seconds


In [12]:
prop_pool = Propagator(circuit)

In [13]:
num_jobs = 8
time_pool_start = time.time()
prop_pool.propagate(pruners=[DeadQubitPruner(), XYWeightPruner()], num_jobs=num_jobs)
time_pool_end = time.time()
print(f'Multiprocess (num_jobs={num_jobs}): {time_pool_end - time_pool_start:.3f} seconds')

Multiprocess (num_jobs=8): 2.441 seconds


In [23]:
print(f'Speedup: {(time_seq_end - time_seq_start)/(time_pool_end - time_pool_start):.3f}x')

Speedup: 5.687x
